# Correlation and spatial autocorrelation

Because relationships with fire are probably non-linear, Pearson correlation is not enough here. This notebook uses:

- **Spearman** for monotonic relationships and for clustering redundant features
- **ROC-AUC per feature** (how well one feature alone separates fire from no fire)
- **Mutual information** (any kind of dependence, including U-shaped ones)
- **Binned fire-rate curves** to see the actual shape
- **Moran's I and a correlogram** for spatial autocorrelation

Set `SEASON` in the first cell and re-run everything to compare summer and winter.
Association is not causation, and your panel is a sample (fire rate about 34%), so read these as
rankings and shapes, not as natural effect sizes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster import hierarchy
from scipy.spatial import cKDTree
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

candidates = [Path("data/processed"), Path("project/data/processed"), Path("../data/processed")]
PROC = next(p for p in candidates if p.exists())
MODEL_DIR = PROC / "panel_same_date_model"
GRID_PATH = PROC / "ca_grid_5km.parquet"

LABEL = "fire"
META = {"date", "pixel_id", "year", "season", "season_year", "season_id", "season_complete",
        "lc_year", "role", "cv_fold"}
NOT_FEATURES = {"lc_sum", "valid_frac", "ndvi_n_images"}   # quality flags, not predictors

# Superseded or flag-like columns. "elevation" is the old centroid value (wrong at the coast and
# nearly identical to elev_mean); dem_land_frac only describes how much of a cell is land.
EXCLUDE = {"elevation", "dem_land_frac"}

# Land cover fractions sum to 1, so single classes are partly redundant and many are rare.
# Collapse them into a few meaningful groups (MODIS IGBP class names). Set to False to use raw classes.
USE_LC_GROUPS = True
LC_GROUPS = {
    "lcg_forest": ["evergreen_needle", "evergreen_broad", "decid_needle", "decid_broad", "mixed_forest"],
    "lcg_shrub": ["closed_shrub", "open_shrub"],
    "lcg_savanna": ["woody_savanna", "savanna"],
    "lcg_grass": ["grassland"],
    "lcg_crop": ["cropland", "crop_mosaic"],
    "lcg_wetland": ["wetland"],
    "lcg_urban": ["urban"],
    "lcg_barren_snow": ["barren", "snow_ice"],
    "lcg_water": ["water"],
}

# ---- choose the subset to explore: "summer", "winter" or "all" ----
SEASON = "summer"
USE_COMPLETE = True      # only complete seasons

df = pd.read_parquet(MODEL_DIR)
df["date"] = pd.to_datetime(df["date"])

# ---- explore TRAINING rows only (the test set must stay unseen) ----
SPLIT_PATH = PROC / "split.parquet"
if SPLIT_PATH.exists():
    split = pd.read_parquet(SPLIT_PATH, columns=["pixel_id", "date", "role", "cv_fold"])
    df = df.merge(split, on=["pixel_id", "date"], validate="1:1")
    df = df[df["role"] == "train"].reset_index(drop=True)
    print("training rows only:", len(df))
else:
    print("WARNING: no split.parquet found, exploring ALL rows (includes future test data)")

# ---- land cover groups ----
raw_lc = [c for c in df.columns if c.startswith("lc_") and c not in ("lc_sum", "lc_year")]
made = []
for g, names in LC_GROUPS.items():
    cols = [f"lc_{n}" for n in names if f"lc_{n}" in df.columns]
    if cols:
        df[g] = df[cols].sum(axis=1)
        made.append(g)
if USE_LC_GROUPS and made:
    EXCLUDE = EXCLUDE | set(raw_lc)
    print(f"land cover: {len(raw_lc)} raw classes -> {len(made)} groups")
elif USE_LC_GROUPS:
    print("no land cover class matched LC_GROUPS (different class names?), using raw classes")
else:
    EXCLUDE = EXCLUDE | set(made)

d = df if SEASON == "all" else df[df["season"] == SEASON]
if USE_COMPLETE:
    d = d[d["season_complete"]]
d = d.reset_index(drop=True)

feats = [c for c in d.columns
         if c not in (META | NOT_FEATURES | EXCLUDE | {LABEL})
         and pd.api.types.is_numeric_dtype(d[c]) and d[c].nunique() > 1]
print(f"{SEASON}: {len(d):,} rows | {d['pixel_id'].nunique():,} pixels | fire rate {d[LABEL].mean():.3f}")
print(f"{len(feats)} numeric features; NaN cells: {int(d[feats].isna().sum().sum())}")
print("features:", feats)

## 1. Feature-feature correlation (Spearman, clustered)
Spearman captures any monotonic relationship. Clustering puts similar features next to each other,

In [ ]:
rho = d[feats].corr(method="spearman").fillna(0)
dist = 1 - rho.abs()
order = hierarchy.leaves_list(hierarchy.linkage(squareform(dist.values, checks=False), "average"))
rho_o = rho.iloc[order, order]

fig, ax = plt.subplots(figsize=(0.32 * len(feats) + 4, 0.32 * len(feats) + 3))
im = ax.imshow(rho_o.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(feats)))
ax.set_xticklabels(rho_o.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(feats)))
ax.set_yticklabels(rho_o.index, fontsize=7)
fig.colorbar(im, shrink=0.7, label="Spearman rho")
plt.show()

iu = np.triu_indices(len(feats), 1)
pairs = pd.DataFrame({"a": rho.index[iu[0]], "b": rho.columns[iu[1]], "rho": rho.values[iu]})
pairs["abs_rho"] = pairs["rho"].abs()
print("highly correlated pairs (|rho| >= 0.8):")
pairs[pairs["abs_rho"] >= 0.8].sort_values("abs_rho", ascending=False).drop(columns="abs_rho")

## 2. Each feature against fire: Spearman, AUC and mutual information
- **Spearman / AUC** only see monotonic relationships (AUC 0.5 means no separation).
- **Mutual information** sees any dependence. A feature with a high MI but low |Spearman| and AUC near 0.5
  is likely **non-monotonic** (for example fire is high at both low and high values).
- `mi_shuffled` is the MI you get from random noise. Features near that value carry no usable signal.

In [ ]:
y = d[LABEL].values
Xf = d[feats].fillna(d[feats].median())

mi = mutual_info_classif(Xf, y, n_neighbors=5, random_state=0)
rng = np.random.default_rng(0)
mi_null = mutual_info_classif(Xf, rng.permutation(y), n_neighbors=5, random_state=0)

rows = []
for c, m in zip(feats, mi):
    x = Xf[c].values
    auc = roc_auc_score(y, x)
    rows.append({"feature": c, "spearman": spearmanr(x, y).correlation, "auc": auc,
                 "auc_strength": abs(auc - 0.5) * 2, "mutual_info": m})
rank = pd.DataFrame(rows).set_index("feature").sort_values("mutual_info", ascending=False)
rank["mi_shuffled"] = pd.Series(mi_null, index=feats).mean()
rank["mi_rank"] = rank["mutual_info"].rank(ascending=False)
rank["lin_rank"] = rank["spearman"].abs().rank(ascending=False)
rank["nonlinear_hint"] = (rank["lin_rank"] - rank["mi_rank"]) >= 5   # MI ranks much higher than Spearman
rank.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(rank["spearman"].abs(), rank["mutual_info"], s=18)
for name, r in rank.head(8).iterrows():
    ax.annotate(name, (abs(r["spearman"]), r["mutual_info"]), fontsize=7)
ax.axhline(rank["mi_shuffled"].iloc[0], ls="--", c="grey", label="MI of shuffled labels")
ax.set_xlabel("|Spearman| with fire")
ax.set_ylabel("mutual information with fire")
ax.legend()
plt.show()

## 3. The shape of the relationship
Fire rate by decile of each top feature (by MI). The dashed line is the overall fire rate.
Curves that rise then fall, or jump at a threshold, are what linear correlation misses.

In [ ]:
top = list(rank.index[:9])
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, c in zip(axes.ravel(), top):
    x = d[c]
    bins = pd.qcut(x, 10, duplicates="drop") if x.nunique() > 12 else x
    g = d.groupby(bins, observed=True)[LABEL].mean()
    xs = d.groupby(bins, observed=True)[c].median()
    ax.plot(xs.values, g.values, marker="o")
    ax.axhline(d[LABEL].mean(), ls="--", c="grey")
    ax.set_title(c, fontsize=9)
    ax.set_ylabel("fire rate")
plt.tight_layout()
plt.show()

## 4. Within-date view
The difference between pixels on the same day.

If the AUC stays high, the feature separates burning from non-burning places. If it falls to 0.5,
the raw signal was mostly the date or season effect.

In [ ]:
dyn = [c for c in feats if d.groupby("pixel_id")[c].nunique().max() > 1]   # varies over time within a pixel
n_per_date = d.groupby("date")[LABEL].transform("size")
w = d[n_per_date >= 2].copy()
for c in dyn:
    w[c] = w[c] - w.groupby("date")[c].transform("mean")
print(f"{len(dyn)} time-varying features; {w['date'].nunique():,} dates with >= 2 rows")

yw = w[LABEL].values
cmp_rows = []
for c in dyn:
    xr = d[c].fillna(d[c].median()).values
    xw = w[c].fillna(0).values
    cmp_rows.append({"feature": c,
                     "auc_raw": roc_auc_score(d[LABEL].values, xr),
                     "auc_within_date": roc_auc_score(yw, xw)})
within = pd.DataFrame(cmp_rows).set_index("feature")
within["strength_raw"] = (within["auc_raw"] - 0.5).abs() * 2
within["strength_within"] = (within["auc_within_date"] - 0.5).abs() * 2
within.sort_values("strength_within", ascending=False).round(3)

## 5. Spatial autocorrelation
Each pixel is reduced to one value (fire rate, or the mean of a feature), and we test whether
nearby pixels have similar values.

- **Moran's I** (k nearest neighbours): near +1 means strong clustering, near 0 is random,
  negative is a checkerboard. The p-value comes from shuffling the values over the pixels.
- Fire rate per pixel is noisy when a pixel has few rows, so only pixels with at least `MIN_ROWS` rows are used,
  and the panel is a sample, so treat the fire-rate result as a property of **this sample**. (Due to the limitation of winter set, `MIN_ROWS` was set as 1 in winter, otherwise only 10 pixels can be examinated.)
- Distances use a flat approximation from lat/lon (fine for exploration).

In [ ]:
grid = pd.read_parquet(GRID_PATH, columns=["pixel_id", "lat", "lon"]).set_index("pixel_id")
pix = d.groupby("pixel_id").agg(rows=(LABEL, "size"), fires=(LABEL, "sum"))
pix["fire_rate"] = pix["fires"] / pix["rows"]
pix = pix.join(d.groupby("pixel_id")[feats].mean()).join(grid)

MIN_ROWS = 3 if SEASON == "summer" else 1
if len(p) < 200:
    print(f"WARNING: only {len(p)} pixels, Moran's I is not reliable") # winter set is too small if the rows are filtered to 3
K = 8
N_PERM = 499
p = pix[(pix["rows"] >= MIN_ROWS)].dropna(subset=["lat", "lon"]).copy()

lat0, lon0 = p["lat"].mean(), p["lon"].mean()
xy = np.c_[(p["lon"] - lon0) * 111.32 * np.cos(np.radians(lat0)), (p["lat"] - lat0) * 110.57]   # km
tree = cKDTree(xy)
_, nbr = tree.query(xy, k=K + 1)
nbr = nbr[:, 1:]                                   # drop the point itself
print(f"{len(p):,} pixels with >= {MIN_ROWS} rows")


def morans_i(z):
    z = z - z.mean()
    den = (z * z).sum()
    return np.nan if den == 0 else (z * z[nbr].mean(axis=1)).sum() / den


rng = np.random.default_rng(0)
rows = []
for col in ["fire_rate"] + feats:
    z = p[col].fillna(p[col].median()).values.astype(float)
    I = morans_i(z)
    null = np.array([morans_i(rng.permutation(z)) for _ in range(N_PERM)])
    rows.append({"variable": col, "morans_I": I, "null_mean": np.nanmean(null),
                 "p_value": (1 + (np.abs(null) >= abs(I)).sum()) / (N_PERM + 1)})
moran = pd.DataFrame(rows).set_index("variable").sort_values("morans_I", ascending=False)
moran.round(4)

In [ ]:
other = "elev_mean" if "elev_mean" in p.columns else feats[0]
fig, axes = plt.subplots(1, 2, figsize=(12, 7))
for ax, col, cmap in [(axes[0], "fire_rate", "Reds"), (axes[1], other, "viridis")]:
    sc = ax.scatter(p["lon"], p["lat"], c=p[col], s=6, cmap=cmap)
    fig.colorbar(sc, ax=ax, shrink=0.7)
    ax.set_title(col)
    ax.set_xlabel("lon")
    ax.set_ylabel("lat")
plt.tight_layout()
plt.show()